# Aufgabe 1: Betrachtung der Daten

In [ ]:
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

df_labeled = pd.read_csv("data/buchungen.csv")
df_unlabeled = pd.read_csv("data/buchungen_test.csv")  # Datensatz ohne Storniert-Status

display(df_labeled.dtypes)  # Ausgabe der Spalten und deren jeweiligen Datentyp
display(df_labeled)

In [ ]:
import matplotlib.pyplot as plt

# TODO: Datenanalyse

# df = df_labeled.copy()

# float(df.loc[df["storniert"] == 1]["storniert"].count() / df["storniert"].count() * 100), 2

# float(
#     df.loc[(df["storniert"] == 1) & (df["hotel_typ"] == "Ferienhotel")]["storniert"].count()
#     / df.loc[df["hotel_typ"] == "Ferienhotel"]["storniert"].count()
#     * 100
# )

# float(
#     df.loc[(df["storniert"] == 1) & (df["hotel_typ"] == "Stadthotel")]["storniert"].count()
#     / df.loc[df["hotel_typ"] == "Stadthotel"]["storniert"].count()
#     * 100
# )

# df["anreisedatum"] = pd.to_datetime(df["anreisedatum"])
# df = df.set_index("anreisedatum").sort_index()

# plt.figure(figsize=(20, 8))
# # Vorlaufzeit, Tages-preis, Warteliste
# plt.plot(df.index, df["durchschn_tagespreis"], color="#9B0000", alpha=1)

# plt.xlabel("Datum")
# plt.grid(alpha=0.5)
# plt.margins(x=0)
# plt.legend()
# plt.tight_layout()
# plt.show()

# Aufgabe 2/3: Vorverarbeitung und Feature Engineering

In [ ]:
# Duplizierte Datenpunkte entfernen (hier vor dem Split)
df_labeled.duplicated().sum()
df_labeled = df_labeled.drop_duplicates()

# Anpassung von Datentypen zur Verarbeitung
df_labeled["anreisedatum"] = pd.to_datetime(df_labeled["anreisedatum"])
df_labeled["stammgast"] = df_labeled["stammgast"].astype(str)  # Stammgast ist kategorisch
df_labeled["agentur_id"] = df_labeled["agentur_id"].astype(str)  # Agenturen sind kategorisch

df_unlabeled["anreisedatum"] = pd.to_datetime(df_unlabeled["anreisedatum"])
df_unlabeled["stammgast"] = df_unlabeled["stammgast"].astype(str)
df_unlabeled["agentur_id"] = df_unlabeled["agentur_id"].astype(str)

In [ ]:
# Anzahl Feiertage im Buchungszeitraum - Vorerst einfach als Idee, keine Ahnung ob sinnvoll
df_holidays = pd.read_csv("data/portugal_feiertage.csv")
df_holidays["datum"] = pd.to_datetime(df_holidays["datum"])
df_holidays = df_holidays.set_index("datum")


def map_holidays(df: pd.DataFrame) -> None:
    df["abreisedatum"] = df["anreisedatum"] + pd.to_timedelta(
        (df["naechte_wochenende"] + df["naechte_wochentag"]).clip(lower=1) - 1, unit="D"
    )

    df["anzahl_feiertage"] = 0
    for d in df_holidays.index:
        df["anzahl_feiertage"] += ((df["anreisedatum"] <= d) & (df["abreisedatum"] >= d)).astype(int)

    df.drop(columns=["abreisedatum"], inplace=True)


map_holidays(df_labeled)
map_holidays(df_unlabeled)

In [ ]:
# Monate / Saison als Feature über anreisedatum

In [ ]:
from sklearn.model_selection import train_test_split

# Entfernen von nicht relevanten Spalten
df_labeled = df_labeled.drop(columns=["anreisedatum", "reservierungsstatus", "reservierungsstatus_datum"])

# Im Folgendem wird nur noch der Trainingsdatensatz betrachtet um Data Leakage zu vermeiden
X_train, X_test, y_train, y_test = train_test_split(
    df_labeled.drop(columns=["storniert"]),
    df_labeled.pop("storniert"),
    test_size=0.2,
    random_state=42,
)

categorical = X_train.select_dtypes(exclude=np.number).columns
numerical = X_train.select_dtypes(include=np.number).columns

In [ ]:
# Korrelationen der numerischen Features
df_corr = X_train[numerical].copy()
df_corr["storniert"] = y_train

correlations = df_corr.corr()
display(correlations["storniert"].sort_values(ascending=False))

In [ ]:
# Spalten die fehlende Werte enthalten ausgeben
missing = X_train.isna().sum()
display(missing[missing > 0])

# Ausreißern veranschaulichen
# Hier vielleicht sinnvoll manche Spalten nicht zu clippen
display(
    X_train.select_dtypes(include=np.number).describe(
        percentiles=[0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 0.995, 0.999]
    )
)

In [ ]:
# Behandlung fehlender Werte und OneHotEncoding
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

categorical_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="constant", fill_value="Keine Angabe")),  # Herkunftsland und Agenturen
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

In [ ]:
# Behandlung fehlender Werte und Ausreißer
from sklearn.preprocessing import FunctionTransformer, StandardScaler


def clip_numericals(X: np.ndarray) -> np.ndarray:
    lower = np.percentile(X, 1, axis=0)
    upper = np.percentile(X, 99, axis=0)

    return np.clip(X, lower, upper)


numerical_pipeline = Pipeline(
    [
        # ("imputer", SimpleImputer(strategy="mean")),
        ("imputer", SimpleImputer(strategy="constant", fill_value=0.0)),  # Kinder (keine Angabe = keine Kinder)
        ("clip", FunctionTransformer(clip_numericals, validate=False)),
        ("scaler", StandardScaler()),  # Für logistische Regression
    ]
)

# Aufgabe 4: Training der Modelle

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score
from sklearn.model_selection import GridSearchCV

preprocessor = ColumnTransformer([("num", numerical_pipeline, numerical), ("cat", categorical_pipeline, categorical)])


def show_metrics(data: pd.DataFrame, pred: pd.DataFrame, title: str = "") -> None:
    tn, fp, fn, tp = confusion_matrix(data, pred).ravel()

    print(
        f"{title}\n"
        f"Accuracy:       {accuracy_score(data, pred):.7f}\n"
        f"Precision:      {precision_score(data, pred):.7f}\n"
        f"Recall:         {recall_score(data, pred):.7f}\n"
        f"False positive: {fp:.7f}\n"
        f"False negative: {fn:.7f}\n"
    )

In [ ]:
from sklearn.tree import DecisionTreeClassifier

grid_search = GridSearchCV(
    estimator=Pipeline([
        ("preprocessor", preprocessor),
        ("tree", DecisionTreeClassifier(random_state=42))
    ]),
    param_grid={
        "tree__max_depth": [5, 15, 30],
        "tree__min_samples_split": [2, 25, 100],
        "tree__min_samples_leaf": [1, 20, 50],
        "tree__criterion": ["gini", "entropy"],
    },
    scoring="accuracy",
    cv=5,
    n_jobs=-1,  # =1 für Ausgabe
    verbose=3,
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_train)
show_metrics(y_train, y_pred, title="Entscheidungsbaum (Trainingsdaten)")

y_pred = best_model.predict(X_test)
show_metrics(y_test, y_pred, title="Entscheidungsbaum (Testdaten)")

print("Beste Parameter:", grid_search.best_params_)

In [ ]:
# Wichtigkeit der Features, auslagern in Funktion
onehot = best_model.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"]
feature_names_cat = onehot.get_feature_names_out(categorical)

feature_names = list(numerical) + feature_names_cat.tolist()

importances = best_model.named_steps["tree"].feature_importances_

df_importances = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

display(df_importances)

In [ ]:
from sklearn.linear_model import LogisticRegression

grid_search = GridSearchCV(
    estimator=Pipeline([
        ("preprocessor", preprocessor),
        ("logistic", LogisticRegression(random_state=42))
    ]),
    param_grid={
        "logistic__C": [0.01, 0.1, 1, 10],
        "logistic__penalty": ["l2"],
        "logistic__solver": ["lbfgs"],
        "logistic__max_iter": [500],
    },
    scoring="accuracy",
    cv=5,
    n_jobs=-1,
    verbose=3,
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_train)
show_metrics(y_train, y_pred, title="Logistische Regression (Trainingsdaten)")

y_pred = best_model.predict(X_test)
show_metrics(y_test, y_pred, title="Logistische Regression (Testdaten)")

print("Beste Parameter:", grid_search.best_params_)

In [ ]:
# Weitere Modelle mal als Vorschrieb

In [ ]:
from sklearn.ensemble import RandomForestClassifier

grid_search = GridSearchCV(
    estimator=Pipeline([
        ("preprocessor", preprocessor),
        ("rf", RandomForestClassifier(random_state=42))
    ]),
    param_grid={
        "rf__n_estimators": [100, 200],
        "rf__max_depth": [10, 20, None],
        "rf__min_samples_split": [2, 10, 50],
        "rf__max_features": ["sqrt", "log2"],
    },
    scoring="accuracy",
    cv=5,
    n_jobs=-1,
    verbose=3
)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

grid_search = GridSearchCV(
    estimator=Pipeline([
        ("preprocessor", preprocessor),
        ("gb", GradientBoostingClassifier(random_state=42))
    ]),
    param_grid={
        "gb__n_estimators": [100, 200],
        "gb__learning_rate": [0.05, 0.1],
        "gb__max_depth": [3, 5],
        "gb__min_samples_split": [2, 10],
        "gb__min_samples_leaf": [1, 5],
        "gb__subsample": [0.8, 1.0]
    },
    scoring="accuracy",
    cv=5,
    n_jobs=-1,
    verbose=3
)

In [ ]:
from sklearn.ensemble import StackingClassifier

grid_search = GridSearchCV(
    estimator=Pipeline([
        ("preprocessor", preprocessor),
        ("stack", StackingClassifier(
            estimators=[
                ("tree", DecisionTreeClassifier(max_depth=10, min_samples_split=10, random_state=42)),
                ("gb", GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, max_depth=3, random_state=42)),
                ("rf", RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42))
            ],
            final_estimator=LogisticRegression(max_iter=500, random_state=42),
            cv=5,
            n_jobs=-1,
        ))
    ]),
    param_grid={
        "stack__final_estimator__C": [0.1, 1, 10],
        "stack__final_estimator__penalty": ["l2"]
    },
    scoring="accuracy",
    cv=5,
    n_jobs=-1,
    verbose=3
)

# Aufgabe 5: Evaluation

# Aufgabe 6: Vorhersage auf unbekannten Daten

In [ ]:
# Hier dann bestes Modell auf kompletten Datensatz trainieren

X = df_labeled.drop(columns=["storniert"])
y = df_labeled.pop("storniert")

In [ ]:
y_pred = best_model.predict(df_unlabeled)

df_unlabeled["storniert_pred"] = y_pred
df_unlabeled[["buchungs_id", "storniert_pred"]].to_csv("data/vorhersage.csv", index=False)